In [1]:
import os

os.chdir("../")

In [2]:
import warnings
from argparse import Namespace
from functools import partial

import numpy as np
import pandas as pd
import torch
from net.net import LSTMClassificationHead, RobertaForRegression
from safetensors.torch import load_file
from sklearn.preprocessing import (
    PowerTransformer,
)
from transformers import (
    AutoConfig,
    AutoTokenizer,
)
from utils.dataset import SMILESDataset
from utils.metrics import compute_metrics
from utils.plotting import process_predictions
from utils.utils import seed_torch

warnings.simplefilter("ignore")


/data/home/hadi/.conda/envs/pt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
seed = 1
seed_torch(seed)

save_model_path = "/data/home/hadi/test/models/ChemBERTa-zinc-base-v1"
tokenizer = AutoTokenizer.from_pretrained(save_model_path)
config = AutoConfig.from_pretrained(save_model_path)

model = RobertaForRegression.from_pretrained(
    save_model_path,
    num_labels=1,
    ignore_mismatched_sizes=True,
    last_use=False,
    weighted=False,
    layers_to_use=[-1, -3, -5],
    use_lora=False,
)

model.classifier = LSTMClassificationHead(
    hidden_dim=768 * 3,
    lstm_hidden_size=768,
    num_labels=1,
    feature_method="mean",
    lstm_layers=1,
    dropout=0.1,
    bidirectional=True,
    skip=False,
    cls_skip=False,
    out_conv=None,
)

best_checkpoint = "models/best_checkpoint/checkpoint-380982/model.safetensors"
model.load_state_dict(load_file(best_checkpoint))


Some weights of RobertaForRegression were not initialized from the model checkpoint at /data/home/hadi/test/models/ChemBERTa-zinc-base-v1 and are newly initialized: ['classifier.head.0.bias', 'classifier.head.0.weight', 'classifier.head.3.bias', 'classifier.head.3.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<All keys matched successfully>

In [6]:
# Load data and apply the same preprocessing as the model

df_train = pd.read_csv("data/train.csv")
df_test = pd.read_csv("data/test.csv")

numeric_columns = ["rt"]

scaler = PowerTransformer(method="box-cox", standardize=True)

df_train[numeric_columns] = scaler.fit_transform(df_train[numeric_columns])
df_test[numeric_columns] = scaler.transform(df_test[numeric_columns])

compute_metrics = partial(compute_metrics, scaler=scaler)

test_dataset = SMILESDataset(
    df_test,
    "smiles",
    "rt",
    tokenizer,
    max_length=89,
    augment=False,
    add_special_tokens=False,
    # weight_column='weights',
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print(f" Number of test samples: {len(test_dataset)}")

 Number of test samples: 7798


In [7]:
# Move model to the device and set it to evaluation mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Lists to store true and predicted values
y_true_list = []
y_pred_list = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        # Move each tensor in the batch to the device
        batch = {k: v.to(device) for k, v in batch.items()}

        labels = batch.get("labels").to(device)

        # Forward pass
        outputs = model(**batch).logits

        # Append true and predicted values
        y_true_list.extend(labels.cpu().numpy())
        y_pred_list.extend(outputs.cpu().numpy())

args = Namespace(path="notebooks/resullt_predict/", model_name="test_set")

process_predictions(y_true_list, y_pred_list, scaler, args)

r2 score: 0.9069486172254642
rmse: 53.84088
MAE: 26.23516
MAPE: 0.032495983


(0.9069486172254642, 26.23516, 0.032495983, 53.84088)

In [8]:
# get the output of the model for one test sample
sample = test_dataset[0]
model.to("cpu")
model.eval()
with torch.no_grad():
    inputs = sample
    outputs = model(
        input_ids=inputs["input_ids"].unsqueeze(0),
        attention_mask=inputs["attention_mask"].unsqueeze(0),
        output_attentions=True,
        labels=inputs["labels"].unsqueeze(0),
    )

    input_unscale = scaler.inverse_transform(np.array(inputs["labels"]).reshape(-1, 1))[0][0]
    result_unscale = scaler.inverse_transform(np.array(outputs.logits).reshape(-1, 1))[0][0]

    print("Actual RT:", input_unscale, "Predicted RT:", result_unscale)


Actual RT: 745.50214 Predicted RT: 760.86755


In [18]:
import pandas as pd

df_test = pd.read_csv("data/test.csv")

results = pd.read_csv("notebooks/resullt_predict/results_test_set.csv")
results["difference"] = abs(results["prediction"] - results["rt"])

# calculate the mean and std of the predictions
results["prediction"].mean(), results["prediction"].std()

print(results["difference"].mean(), results["difference"].std())

# show predictions with high error more than 3 std
# outlier_predictions = results.query("abs(prediction-rt)>2*(rt).std()").sort_values("difference", ascending=False)
outlier_predictions = results.query("abs(prediction-rt)>3*(difference).std()").sort_values(
    "difference", ascending=False
)

# print the number of outliers
print(len(outlier_predictions))

# save the outliers to a csv file
display(outlier_predictions)
df_test["prediction"] = results["prediction"]
df_test.loc[outlier_predictions.index].to_csv("notebooks/resullt_predict/3sigma_outliers.csv", index=False)


26.23515946652988 47.01957211995218
197


,rt,prediction,difference
6628,1429.5905,634.59894,794.99156
4411,1425.0175,701.10420,723.91330
2122,1350.3984,648.37787,702.02053
4222,700.8009,1257.31420,556.51330
5977,809.1044,1365.00090,555.89650
...,...,...,...
352,982.9993,840.06067,142.93863
5851,556.7993,698.58170,141.78240
1834,878.1042,1019.61145,141.50725
7208,611.9023,753.25165,141.34935
